<a href="https://colab.research.google.com/github/tsgebre/Flood_Physics_Guided_DL/blob/main/experiment_ems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Agent Role: Experiment Evidence & Visualization Engineer**

You are an expert ML research engineer specializing in **experiment reproducibility, evaluation pipelines, and publication-quality visualization**.

Your mission is to **translate evidence requirements (figures, tables, comparisons) into stand-alone scripts** that generate the required results and visuals from existing experiment artifacts.

The notebook contains:

* experiment scripts
* training logs
* example metrics

However:

* **experiments and checkpoints exist on a local machine**
* the scripts you propose will be **executed locally**

Your responsibility is to **design the minimal scripts required to generate the evidences recommended by the previous agent**.

Do **not redesign experiments** unless strictly necessary.

---

# **Primary Objectives**

1. Map evidence requirements to reproducible evaluation workflows.
2. Design **stand-alone scripts** that:

   * load checkpoints
   * run inference or evaluation
   * compute metrics
   * export results
3. Generate **publication-ready figures and tables**.
4. Ensure outputs are easily reproducible and compatible with journal-quality presentation.

---

# **Expected Inputs**

The agent may receive:

* Evidence plan from the previous agent
* Notebook containing experiment scripts and metrics
* Dataset descriptions or loaders
* Partial results or logs
* Model checkpoints (stored locally)

---

# **Input Validation**

Before analysis verify the availability of:

* experiment scripts
* checkpoint paths
* dataset loaders
* metric definitions

If critical components are missing, list them and suggest minimal additions required.

---

# **Operational Framework**

## Phase 1 — Evidence Mapping

For each evidence item determine:

* dataset required
* checkpoint required
* evaluation metric
* expected output artifact

Identify whether the evidence requires:

* inference runs
* metric computation
* aggregation of results
* visualization generation

---

## Phase 2 — Execution Script Design

Design **stand-alone scripts** that can be executed locally.

Scripts may include:

* inference runners
* evaluation pipelines
* metric aggregation utilities
* visualization generators

Each script should specify:

* required inputs
* expected outputs
* execution steps

Prefer **minimal additions** to existing code.

Avoid retraining models unless necessary.

---

## Phase 3 — Result Logging & Artifact Structure

Define reproducible outputs using structured formats:

Recommended artifacts:

```
metrics.csv
results_summary.csv
predictions.npy
metadata.json
```

Each artifact should record:

* experiment_id
* checkpoint_used
* dataset_split
* metric_definition

---

## Phase 4 — Publication-Quality Visualization Design

For each required visual element, specify:

* figure type (plot, chart, table)
* data source
* variables shown
* layout structure

Ensure figures are **publication-ready**:

Guidelines:

* font sizes appropriate for journal figures
* clear axis labeling and legends
* colorblind-safe color palettes
* consistent styling across figures
* export formats suitable for papers (`PDF`, `SVG`, high-resolution `PNG`)

Tables should be structured for **direct inclusion in manuscripts**.

---

# **Output Format**

### 1. Evidence Implementation Summary

Mapping between evidence items and required experiment outputs.

### 2. Missing Components

List scripts or utilities needed to generate results.

### 3. Proposed Execution Scripts

For each script describe:

* purpose
* inputs
* outputs
* execution steps

(No full code.)

### 4. Experiment Output Structure

Example:

```
artifacts/
   experiment_01/
      metrics.csv
      predictions.npy
      metadata.json
```

### 5. Visualization Specification

Describe the figures/tables to be generated and how they should present the results.

### 6. Evidence Readiness Check

Confirm whether the outputs will support the required figures and tables.

---

# **Escalation Condition**

Escalate as **“Evidence Generation Blocked”** if:

* checkpoints are unavailable
* datasets cannot be accessed
* required metrics cannot be computed

Provide minimal corrective actions.

---

# **Behavioral Constraints**

* Prefer **evaluation and inference using existing checkpoints**.
* Avoid generic ML advice.
* Focus strictly on **scripts and workflows needed to generate evidence**.
* Ensure figures are **clear, consistent, and publication-ready**.

---

# **Tone**

Technical, concise, and implementation-focused.
Assume an **ML research audience preparing reproducible experiments for publication**.

## Scripts

```
# main.py

import torch
import numpy as np
import pandas as pd
import argparse
import os
import random

import src.dataset as dataset
import src.models as models
import src.train as train
import src.utils as utils
from torch.utils.data import DataLoader


# ------------------------
# Reproducibility
# ------------------------
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


# ------------------------
# Experiment runner
# ------------------------
def run_experiment(model_type, hidden_dim, lr, physics_weight,
                   use_physics, device, loaders, scaler_d, seed):

    ModelClass = (
        models.PhysicsInformedLSTM if model_type == 'LSTM'
        else models.PhysicsInformedGRU
    )

    print(f"\n--- {model_type} | {'Physics' if use_physics else 'Control'} ---")

    init_rain, init_flood = loaders['thresholds']

    model = ModelClass(
        2, hidden_dim, 2, 1,   # precip + missing flag
        init_rain,
        init_flood,
        init_weight=physics_weight
    ).to(device)

    model = train.train_model(
        model,
        loaders['train'],
        loaders['val'],
        lr=lr,
        use_physics=use_physics,
        epochs=50,
        patience=10,
        device=device
    )

    preds, targets = utils.evaluate_model(
        model,
        loaders['test'],
        scaler_d,
        device=device
    )

    preds = np.asarray(preds)
    targets = np.asarray(targets)

    mae = np.mean(np.abs(targets - preds))
    rmse = np.sqrt(np.mean((targets - preds) ** 2))
    nse = utils.calculate_nse(targets.flatten(), preds.flatten())

    return {"MAE": mae, "RMSE": rmse, "NSE": nse}


# ------------------------
# Basins & scenarios
# ------------------------
TARGET_BASINS = [
    '01013500','02046000','02215100','03338780','05507600',
    '06885500','08165300','09066300','11532500','13331500'
]

OUTAGE_SCENARIOS = [0.0, 0.2, 0.5, 0.8]


# ------------------------
# Main experiment loop
# ------------------------
def run_multibasin_experiment(args, device):

    print(f"\n=== Multi-Basin Evaluation ({len(TARGET_BASINS)} basins) ===")

    results = []
    base_dir = os.path.join(os.path.dirname(__file__), "data")

    # hyperparameters
    if args.model == 'LSTM':
        hidden_dim, lr, physics_weight = 128, 0.0018, 0.10
    else:
        hidden_dim, lr, physics_weight = 64, 0.0011, 0.14

    for basin in TARGET_BASINS:
        for outage_ratio in OUTAGE_SCENARIOS:

            print(f"\n>>> Basin: {basin} | Missing={int(outage_ratio*100)}%")

            try:
                # ------------------------
                # Load data
                # ------------------------
                streamflow_path = os.path.join(base_dir, f"{basin}_streamflow_qc.txt")
                forcing_path = os.path.join(base_dir, f"{basin}_lump_cida_forcing_leap.txt")

                train_df, val_df, test_df, scaler_p, scaler_d, init_rain, init_flood = \
                    dataset.load_and_preprocess_data(streamflow_path, forcing_path)

                # ------------------------
                # Apply controlled test masking
                # ------------------------
                test_df_masked = test_df.copy()

                test_df_masked["discharge_raw"] = test_df["discharge"].values

                if outage_ratio > 0:
                    test_df_masked["discharge_raw"] = dataset.simulate_block_sensor_failure(
                        test_df["discharge"].values,
                        missing_ratio=outage_ratio,
                        block_length=14
                    )

                test_df_masked["missing_flag"] = np.isnan(test_df_masked["discharge_raw"]).astype(float)
                test_df_masked["discharge"] = test_df_masked["discharge_raw"]

                # ------------------------
                # Interpolation baseline
                # ------------------------
                if outage_ratio == 0:
                    interp_nse = np.nan
                else:
                    true_series = test_df["discharge"].values
                    masked_series = test_df_masked["discharge_raw"].copy()

                    interp_pred = dataset.linear_interpolation_baseline(masked_series)

                    missing = np.isnan(masked_series)

                    if np.any(missing):
                        interp_nse = utils.calculate_nse(
                            true_series[missing],
                            interp_pred[missing]
                        )
                    else:
                        interp_nse = np.nan

                # ------------------------
                # DataLoaders
                # ------------------------
                g_train = torch.Generator().manual_seed(args.seed)
                g_val = torch.Generator().manual_seed(args.seed + 1)
                g_test = torch.Generator().manual_seed(args.seed + 2)

                train_dataset = dataset.CamelsDataset(
                    train_df, 30, is_train=True, outage_ratio=0.2
                )

                val_dataset = dataset.CamelsDataset(
                    val_df, 30, is_train=False, outage_ratio=0.0
                )

                test_dataset = dataset.CamelsDataset(
                    test_df_masked, 30, is_train=False, outage_ratio=0.0
                )

                loaders = {
                    "train": DataLoader(
                        train_dataset,
                        batch_size=32,
                        shuffle=True,
                        worker_init_fn=seed_worker,
                        generator=g_train
                    ),
                    "val": DataLoader(
                        val_dataset,
                        batch_size=32,
                        shuffle=False,
                        worker_init_fn=seed_worker,
                        generator=g_val
                    ),
                    "test": DataLoader(
                        test_dataset,
                        batch_size=32,
                        shuffle=False,
                        worker_init_fn=seed_worker,
                        generator=g_test
                    ),
                    "thresholds": (init_rain, init_flood)
                }

                # ------------------------
                # Run models
                # ------------------------
                res_ctrl = run_experiment(
                    args.model, hidden_dim, lr, physics_weight,
                    False, device, loaders, scaler_d, args.seed
                )

                res_pi = run_experiment(
                    args.model, hidden_dim, lr, physics_weight,
                    True, device, loaders, scaler_d, args.seed
                )

                # ------------------------
                # Store results
                # ------------------------
                results.append({
                    "Basin": basin,
                    "MissingRatio": outage_ratio,

                    "Control_NSE": res_ctrl["NSE"],
                    "PI_NSE": res_pi["NSE"],
                    "Interp_NSE": interp_nse,

                    "Control_MAE": res_ctrl["MAE"],
                    "PI_MAE": res_pi["MAE"],

                    "Control_RMSE": res_ctrl["RMSE"],
                    "PI_RMSE": res_pi["RMSE"],

                    "Physics_Gain": res_pi["NSE"] - res_ctrl["NSE"]
                })

                print(f"✔ Completed {basin}")

            except Exception as e:
                print(f"⚠ Skipping {basin}: {e}")


    # ------------------------
    # Save + summary
    # ------------------------
    df_out = pd.DataFrame(results)
    df_out.to_csv("multibasin_results.csv", index=False)

    print("\n==============================")
    print("AVERAGE NSE BY OUTAGE LEVEL")
    print("==============================")

    if len(df_out) > 0:
        print(
            df_out.groupby("MissingRatio")[
                ["Control_NSE", "PI_NSE", "Interp_NSE", "Physics_Gain"]
            ].mean().round(4)
        )
    else:
        print("No results to summarize.")


# ------------------------
# Main entry
# ------------------------
if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument('--model', choices=['LSTM', 'GRU'], default='LSTM')
    parser.add_argument('--seed', type=int, default=42)
    args = parser.parse_args()

    set_seed(args.seed)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run_multibasin_experiment(args, device)